# Stage1 XSD−SOXX 行业内部广度来源映射

固定 20 个源交易日总收益差；只检查公司行动、Qlib Stage1 日历、训练期与原 39 项相关性，不读取标签或拟合。

In [1]:
import os,json,hashlib,math
from pathlib import Path
from datetime import datetime,timezone
from zoneinfo import ZoneInfo
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_xsd_semiconductor_breadth_map_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
spec=plan['source'];src=root/spec['card'];assert sha(src/'summary.json')==spec['summary_sha256'];source_summary=json.loads((src/'summary.json').read_text());assert source_summary['NAS_archive_sha256']==spec['NAS_archive_sha256'] and source_summary['NAS_remote_and_scoped_restore_verified']
ny=ZoneInfo('America/New_York');series={};diagnostics={}
for symbol in ('XSD','SOXX'):
 path=root/spec['raw_root']/(symbol+'.json');assert sha(path)==spec['raw_sha256'][symbol]
 obj=json.loads(path.read_text())['chart'];assert obj['error'] is None and len(obj['result'])==1;r=obj['result'][0];assert r['meta']['symbol']==symbol and r['meta']['exchangeTimezoneName']=='America/New_York'
 dates=pd.DatetimeIndex([datetime.fromtimestamp(x,timezone.utc).astimezone(ny).date() for x in r['timestamp']]);assert dates.is_unique and dates.is_monotonic_increasing
 quote=r['indicators']['quote'][0];close=pd.Series(quote['close'],index=dates,dtype=float);adj=pd.Series(r['indicators']['adjclose'][0]['adjclose'],index=dates,dtype=float);assert close.notna().all() and adj.notna().all() and (close>0).all() and (adj>0).all()
 events=r.get('events') or {};div=pd.Series(0.0,index=dates);divdates=[]
 for e in (events.get('dividends') or {}).values():
  d=pd.Timestamp(datetime.fromtimestamp(e['date'],timezone.utc).astimezone(ny).date());assert d in div.index and e['amount']>0 and d not in divdates;div.loc[d]=float(e['amount']);divdates.append(d)
 splits=[]
 for e in (events.get('splits') or {}).values():
  d=pd.Timestamp(datetime.fromtimestamp(e['date'],timezone.utc).astimezone(ny).date());assert d in close.index;splits.append({'date':str(d.date()),'ratio':e['splitRatio']})
 gross=(close+div)/close.shift(1);assert (gross.iloc[1:]>0).all();logret=np.log(gross);adjlog=np.log(adj/adj.shift(1));gap=(logret-adjlog).abs()*10000
 maxgap=float(gap.iloc[1:].max());splitmax=max([abs(float(logret.loc[pd.Timestamp(x['date'])])) for x in splits],default=0)
 diagnostics[symbol]={'rows':len(dates),'first_date':str(dates[0].date()),'last_date':str(dates[-1].date()),'dividend_events':len(divdates),'split_events':splits,'max_abs_gross_adj_log_gap_bp':maxgap,'median_abs_gross_adj_log_gap_bp':float(gap.iloc[1:].median()),'split_day_max_abs_gross_log':splitmax,'all_positive_finite':bool(np.isfinite(logret.iloc[1:]).all())}
 series[symbol]=pd.DataFrame({'close':close,'adjclose':adj,'dividend':div,'log_total_return':logret})
assert series['XSD'].index.equals(series['SOXX'].index)
source_dates=series['XSD'].index
report={'status':'source_preflight_pending_gates','plan_sha256':sha(card/'plan.json'),'source_summary_sha256':spec['summary_sha256'],'source_NAS_sha256':spec['NAS_archive_sha256'],'symbols':diagnostics,'source_pair_dates_exact':True,'model_fits':0,'new_native_accounts':0}
(card/'preflight_result.json').write_text(json.dumps(report,indent=2)+'\n')
assert all(x['max_abs_gross_adj_log_gap_bp']<=plan['quality']['gross_vs_retrospective_adjclose_log_return_abs_gap_max_bp'] and x['split_day_max_abs_gross_log']<=plan['quality']['split_day_gross_log_abs_max'] for x in diagnostics.values()),'corporate action return consistency gate failed'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-07',freq='day'))
parent=plan['parent'];pc=root/parent['card'];art=pc/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'
for name,key in [('models/base_outer_71','base_model_artifact_sha256'),('inputs/features','base_features_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256')]:assert sha(art/name)==parent[key]
with R.uri_context(str(pc/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns) and base.index.equals(frame.index)
train=calendar[(calendar>=pd.Timestamp(plan['train_signal'][0]))&(calendar<=pd.Timestamp(plan['train_signal'][1]))];held=calendar[(calendar>=pd.Timestamp(plan['held_reused_signal'][0]))&(calendar<=pd.Timestamp(plan['held_reused_signal'][1]))]
assert len(train)==plan['quality']['stage1_calendar_rows_train'] and len(held)==plan['quality']['stage1_calendar_rows_held']
assert frame.index[:len(calendar)].equals(calendar) and all(x in source_dates for x in calendar)
soxx_q=D.features(['SOXX'],['$close'],start_time='2011-01-03',end_time='2026-10-07',freq='day')
soxx_q_close=soxx_q['$close'].droplevel('instrument').sort_index();soxx_q_close.index=pd.DatetimeIndex(soxx_q_close.index).normalize();soxx_y_close=series['SOXX']['close'].reindex(soxx_q_close.index)
qret=np.log(soxx_q_close/soxx_q_close.shift(1));yret=series['SOXX']['log_total_return'].reindex(soxx_q_close.index)
valid=qret.notna()&yret.notna();soxx_q_diag={'rows':int(len(soxx_q_close)),'paired_returns':int(valid.sum()),'median_abs_log_return_gap_bp':float(((qret-yret).abs()*10000).loc[valid].median()),'max_abs_log_return_gap_bp':float(((qret-yret).abs()*10000).loc[valid].max())}
coordinate=(series['XSD']['log_total_return']-series['SOXX']['log_total_return']).rolling(20,min_periods=20).sum().rename('XSD_minus_SOXX_log_total_return_20');mapped=coordinate.reindex(calendar)
assert mapped.notna().all() and np.isfinite(mapped).all()
train_coord=mapped.loc[train];held_coord=mapped.loc[held]
train_base=base.loc[train,base_names].astype(float);corr=train_base.corrwith(train_coord).sort_values(key=lambda x:x.abs(),ascending=False);finite=corr.dropna();maxabs=float(finite.abs().max());dup=maxabs>=0.95
static=pd.concat({'feature':mapped.to_frame(),'meta':pd.DataFrame({'source_known':mapped.notna()},index=calendar)},axis=1);static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['train_signal']),'held':tuple(plan['held_reused_signal'])});tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta']);assert len(tr)==len(train) and len(he)==len(held)
report.update(status='source_map_and_training_redundancy_preflight_passed_no_fit',Qlib_calendar_rows=len(calendar),training_rows=len(train),held_rows=len(held),candidate_train_unknown=int(train_coord.isna().sum()),candidate_held_unknown=int(held_coord.isna().sum()),source_feature_min_train=float(train_coord.min()),source_feature_max_train=float(train_coord.max()),parent_original_features=39,top_training_correlations=[{'feature':str(k),'pearson':float(v)} for k,v in corr.head(10).items() if np.isfinite(v)],max_abs_training_pairwise_pearson=maxabs,duplicate_pairwise_ge_095=bool(dup),Qlib_SOXX_close_return_plane_diagnostic=soxx_q_diag,DatasetH_train_rows=len(tr),DatasetH_held_rows=len(he),historical_corporate_action_vintage_qualified=False)
(card/'preflight_result.json').write_text(json.dumps(report,indent=2)+'\n')
print({k:report[k] for k in ('status','training_rows','held_rows','candidate_train_unknown','candidate_held_unknown','max_abs_training_pairwise_pearson','duplicate_pairwise_ge_095')})


2026/10/23 04:08:33 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[77523:MainThread](2026-10-23 04:08:33,792) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[77523:MainThread](2026-10-23 04:08:33,794) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[77523:MainThread](2026-10-23 04:08:33,794) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[77523:MainThread](2026-10-23 04:08:35,071) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[77523:MainThread](2026-10-23 04:08:35,071) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[77523:MainThread](2026-10-23 04:08:35,072) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'status': 'source_map_and_training_redundancy_preflight_passed_no_fit', 'training_rows': 3020, 'held_rows': 944, 'candidate_train_unknown': 0, 'candidate_held_unknown': 0, 'max_abs_training_pairwise_pearson': 0.18058740374007723, 'duplicate_pairwise_ge_095': False}


## Recorder 与 NAS 写入

仅在来源、日历和 DatasetH 预检通过后执行。

In [2]:
import shlex,subprocess,tarfile,tempfile
from src.data.nas_market_data import archive_nas_recorder,_settings
write_spec=json.loads((card/'recorder_write_plan.json').read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());preflight_source=''.join(notebook['cells'][1]['source']);write_source=''.join(notebook['cells'][3]['source'])
assert write_spec['frozen'] and write_spec['plan_sha256']==sha(card/'plan.json') and write_spec['preflight_result_sha256']==sha(card/'preflight_result.json') and write_spec['preflight_cell_sha256']==hashlib.sha256(preflight_source.encode()).hexdigest() and write_spec['write_cell_sha256']==hashlib.sha256(write_source.encode()).hexdigest()
assert report==json.loads((card/'preflight_result.json').read_text()) and report['status']=='source_map_and_training_redundancy_preflight_passed_no_fit'
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists()
state={'status':'started','plan_sha256':sha(card/'plan.json'),'preflight_result_sha256':sha(card/'preflight_result.json'),'model_fits':0,'native_accounts':0,'parent_Recorder_object_loads_total_prior_and_current':12}
state_path.write_text(json.dumps(state,indent=2)+'\n')
with R.start(experiment_name=card.name,uri=str(tracking)):
 recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));state_path.write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True)
 recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,source_identity=spec)
 source_daily=pd.DataFrame({'xsd_close':series['XSD']['close'],'xsd_dividend':series['XSD']['dividend'],'xsd_log_return':series['XSD']['log_total_return'],'soxx_close':series['SOXX']['close'],'soxx_dividend':series['SOXX']['dividend'],'soxx_log_return':series['SOXX']['log_total_return']})
 mapped_signals=pd.DataFrame({'XSD_minus_SOXX_log_total_return_20':mapped,'source_known':mapped.notna()},index=calendar)
 recorder.save_objects(artifact_path='analysis',source_daily=source_daily,mapped_signals=mapped_signals,training_feature_correlations=corr,source_diagnostics=report)
 recorder.log_metrics(train_known=len(train),held_known=len(held),max_abs_training_pairwise_pearson=report['max_abs_training_pairwise_pearson'],model_fits=0,native_accounts=0)
 with R.uri_context(str(tracking)):
  reread=R.get_recorder(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id)).load_object('analysis/mapped_signals')
 assert reread.equals(mapped_signals) and len(reread)==3964
summary={'experiment':card.name,'plan_sha256':sha(card/'plan.json'),'decision':'source_map_passed_retrospective_action_vintage_pending_no_fit' if not report['duplicate_pairwise_ge_095'] else 'source_map_pairwise_duplicate_close_before_fit','objective_achieved':False,'training_rows':len(train),'held_rows':len(held),'candidate_unknown_train_held':[report['candidate_train_unknown'],report['candidate_held_unknown']],'max_abs_training_pairwise_pearson':report['max_abs_training_pairwise_pearson'],'top_training_correlations':report['top_training_correlations'],'corporate_action_diagnostics':report['symbols'],'Qlib_SOXX_close_return_plane_diagnostic':report['Qlib_SOXX_close_return_plane_diagnostic'],'historical_corporate_action_vintage_qualified':False,'historical_independence':False,'model_fits':0,'new_native_accounts':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'source_NAS_sha256':spec['NAS_archive_sha256'],'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='xsd-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  names=[m.name for m in source.getmembers() if m.isfile()]
  assert any(name.endswith('/artifacts/analysis/mapped_signals') for name in names) and any(name.endswith('/artifacts/inputs/dataset') for name in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];state_path.write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'recorder_id':state['recorder_id'],'NAS_archive_sha256':evidence['sha256'],'model_fits':0,'native_accounts':0})


[77523:MainThread](2026-10-23 04:08:35,082) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_xsd_semiconductor_breadth_map_20261023.


[77523:MainThread](2026-10-23 04:08:35,083) INFO - qlib.workflow - [exp.py:258] - Experiment 541018705945447568 starts running ...


[77523:MainThread](2026-10-23 04:08:35,156) INFO - qlib.workflow - [recorder.py:345] - Recorder f3eb1079959b448bb826eb9eda08264a starts running under Experiment 541018705945447568 ...


[77523:MainThread](2026-10-23 04:08:36,118) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'source_map_passed_retrospective_action_vintage_pending_no_fit', 'recorder_id': 'f3eb1079959b448bb826eb9eda08264a', 'NAS_archive_sha256': 'cd9b4920f0b0e33f899a114f107095fbd460ade187f67afb1737fd61e21122b3', 'model_fits': 0, 'native_accounts': 0}


# Stage1 XSD−SOXX 行业内部领导结构来源映射

结论：来源映射通过；可用于单独冻结的回溯历史信息检验，但当前公司行动首次公开与修订时钟未获证，不能进入 0bp 账户或晋升。

使用冻结 Stage1 的 Qlib 日历与原 39 项输入，XSD−SOXX 过去 20 个已完成源交易日的总收益差覆盖全部 3,020 个训练日和 944 个复用历史留出日，缺失均为 0。训练期与原 39 项最大的单项绝对 Pearson 相关为 0.181，对应 `high_chop_pressure`；这只排除了明显单项同义，不证明新增预测信息。

XSD 2015-09-11 2:1 拆分、SOXX 2024-03-07 3:1 拆分；源收盘和除息额重建的单日对数总收益与同快照调整收盘收益最大差分别为 0.880bp、1.299bp。拆分日收益未出现机械倍数跳变。

第一次预检把父来源尾部 2026-10-08/09 两天错误要求包含在固定截至 2026-10-07 的留出日历里；保留了失败凭据，机械修复仅按固定 Qlib 日历切片，不改公式或样本。

原生 Qlib Recorder：`f3eb1079959b448bb826eb9eda08264a`；NAS 远端归档与回读 SHA-256：`cd9b4920f0b0e33f899a114f107095fbd460ade187f67afb1737fd61e21122b3`。0 拟合、0 账户，未读取未来标签。

下一步：冻结原 39 项基线 + 一个 XSD−SOXX 坐标的 H5 SOXX 现金相对信息检验；若信息门失败关闭此版，不改符号或窗口救结果。即使信息门通过，历史公司行动版本仍需时点认证后才能单独设计 0bp 费用后 Stage1 账户。